# 08 — 静态图节点分类：MLP、GCN、GraphSAGE 与 GAT

本课把时间因素拿掉，单独回答：图结构何时提供了节点特征之外的信息？Cora 中节点是论文、边是引用、特征是词袋、标签是研究主题。实验只用 validation 做选择，test 留到最后。

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.citation import load_planetoid, identity_edges, shuffled_edges
from crosscity.models.static_gnn import MLPNodeClassifier, GCN, GraphSAGE, GAT
from crosscity.training.node_classification import train_node_classifier
from crosscity.utils import seed_everything
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. 下载并检查 Cora

首次运行需要网络。PyTorch Geometric 的 `Planetoid` 数据集类负责下载、处理与缓存；`data/raw/` 已被 Git 忽略。

In [ ]:
data = load_planetoid(ROOT / 'data/raw/planetoid', 'cora')
print('nodes/features/classes:', data.num_nodes, data.num_features, data.num_classes)
print('directed edge entries:', data.edge_index.shape[1])
print('train/val/test:', *(int(m.sum()) for m in [data.train_mask, data.val_mask, data.test_mask]))
print('feature row sums:', data.x.sum(1).min().item(), data.x.sum(1).max().item())

## 2. 三种图对照

`identity` 完全切断论文之间的信息；`real` 保留真实引用；`shuffled` 保留图的拓扑和度分布，却打乱节点语义。MLP 不读取任何边，因此只需运行一次。

In [ ]:
graphs = {
    'identity': identity_edges(data.num_nodes),
    'real': data.edge_index,
    'shuffled': shuffled_edges(data.edge_index, data.num_nodes, seed=42),
}
pd.DataFrame({name: {'edges': edges.shape[1], 'max node': int(edges.max())} for name, edges in graphs.items()}).T

## 3. Smoke run：先证明完整链路能运行

下面只跑 5 epoch，不据此下结论。注意所有模型输出均为 `[节点数, 类别数]`。

In [ ]:
smoke_data = replace(data, edge_index=graphs['real']).to(device)
for model_class in [MLPNodeClassifier, GCN, GraphSAGE, GAT]:
    seed_everything(42)
    model = model_class(data.num_features, 64, data.num_classes).to(device)
    result = train_node_classifier(model, smoke_data, max_epochs=5, patience=5)
    print(model_class.__name__, 'shape=', tuple(model(smoke_data.x, smoke_data.edge_index).shape),
          'val=', round(result.validation_accuracy, 3))

## 4. 正式比较

先固定单一种子理解机制；要报告结论时再改为多个种子并报告均值和标准差。这里仍记录 test，但不要根据 test 返回修改模型。

In [ ]:
model_classes = {'MLP': MLPNodeClassifier, 'GCN': GCN, 'GraphSAGE': GraphSAGE, 'GAT': GAT}
records, histories = [], {}
for model_name, model_class in model_classes.items():
    graph_names = ['real'] if model_name == 'MLP' else list(graphs)
    for graph_name in graph_names:
        seed_everything(42)
        experiment = replace(data, edge_index=graphs[graph_name]).to(device)
        model = model_class(data.num_features, 64, data.num_classes).to(device)
        result = train_node_classifier(model, experiment, max_epochs=300, patience=50)
        records.append({'model': model_name, 'graph': graph_name, 'best_epoch': result.best_epoch,
                        'validation_accuracy': result.validation_accuracy, 'test_accuracy': result.test_accuracy})
        histories[(model_name, graph_name)] = result.history
results = pd.DataFrame(records).sort_values('validation_accuracy', ascending=False)
results.style.format({'validation_accuracy': '{:.3f}', 'test_accuracy': '{:.3f}'})

In [ ]:
for key, history in histories.items():
    frame = pd.DataFrame(history)
    plt.plot(frame.epoch, frame.val_accuracy, label='/'.join(key))
plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.legend(ncol=2); plt.show()

## 5. 解释清单

1. MLP 与 identity-GNN 是否接近？若不同，是深度、参数量还是优化造成？
2. real 是否稳定优于 shuffled？只有这一项能表明边与节点语义的对应关系有用。
3. GAT 是否一定胜过均值聚合？复杂模型没有自动优势。
4. 多个种子是否给出相同排序？
5. 下一课将把同一思想迁移到整图分类，并区分 transductive 与 inductive 学习。